# Demucs htdemucs — Оценка по MUSDB18-7

Качественный анализ разделения музыкальных источников с использованием модели Demucs на тестовом наборе данных MUSDB18-7.

## Установка

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import musdb
import soundfile as sf
import torch

sys.path.insert(0, str(Path.cwd()))
from app.separator import DemucsSeparator

STEMS = ["vocals", "drums", "bass", "other"]
STEMS_RU = {"vocals": "Вокал", "drums": "Барабаны", "bass": "Бас", "other": "Остальное"}
STEMS_COLOR = {"vocals": "#e74c3c", "drums": "#3498db", "bass": "#2ecc71", "other": "#9b59b6"}

## 1. Визуализируем дорожку

Загрузите образец дорожки и визуализируйте его форму волны и спектрограмму.

In [ ]:
DATASET_ROOT = "C:/Users/Arina/MUSDB18/MUSDB18-7"
mus = musdb.DB(root=DATASET_ROOT, subsets="test")
track = list(mus.tracks)[0]  # AM Contra - Heart Peripheral
sr = int(track.rate)
mixture = track.audio  # (сэмплы, каналы)
print(f"Track: {track.name}")
print(f"Sample rate: {sr} Hz")
print(f"Duration: {mixture.shape[0] / sr:.2f} s")
print(f"Shape: {mixture.shape}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

# Форма сигнала (левый канал)
t = np.arange(len(mixture)) / sr
ax[0].plot(t, mixture[:, 0], linewidth=0.5, color="#3498db")
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("Amplitude")
ax[0].set_title("Mixture Waveform (Left Channel)")

# Спектрограмма
D = librosa.amplitude_to_db(np.abs(librosa.stft(mixture[:, 0], sr=sr)), ref=np.max)
img = librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="hz", ax=ax[1], cmap="magma")
ax[1].set_title("Mixture Spectrogram")
plt.colorbar(img, ax=ax[1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

## 2. Примените разделение Demucs

In [ ]:
separator = DemucsSeparator(model_name="htdemucs", device="cpu", chunk_size=None)

import tempfile
tmp = tempfile.mktemp(suffix=".wav")
sf.write(tmp, mixture, sr)

import time
start = time.time()
result = separator.separate(tmp)
elapsed = time.time() - start
print(f"Separated in {elapsed:.2f}s")
print(f"Stems: {list(result.stems.keys())}")

import os
os.unlink(tmp)

In [ ]:
# Визуализируйте разделенные стемы
fig, axes = plt.subplots(4, 2, figsize=(14, 10))

for idx, stem in enumerate(STEMS):
    est = result.stems[stem]
    gt = track.targets[stem].audio.astype(np.float32)
    
    if est.ndim == 1:
        est = est[np.newaxis, :]
    est_mono = est[0] if est.shape[0] > 0 else est
    gt_mono = gt[:, 0]
    
    # Форма волны
    t = np.arange(len(est_mono)) / sr
    axes[idx, 0].plot(t, est_mono, linewidth=0.5, color=STEMS_COLOR[stem])
    axes[idx, 0].set_ylabel("Amplitude")
    axes[idx, 0].set_title(f"{STEMS_RU[stem]} — Estimated")
    
    # Спектрограмма
    D = librosa.amplitude_to_db(np.abs(librosa.stft(est_mono, sr=sr)), ref=np.max)
    img = librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="hz", ax=axes[idx, 1], cmap="magma")
    axes[idx, 1].set_title(f"{STEMS_RU[stem]} — Spectrogram")
    plt.colorbar(img, ax=axes[idx, 1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

## 3. Сравнить с эталонными данными

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 8))

for idx, stem in enumerate(STEMS):
    est = result.stems[stem]
    gt = track.targets[stem].audio.astype(np.float32)
    
    if est.ndim == 1:
        est = est[np.newaxis, :]
    est_mono = est[0]
    gt_mono = gt[:, 0]
    
    # Обрезать до одинаковой длины
    n = min(len(est_mono), len(gt_mono))
    t = np.arange(n) / sr
    
    axes[idx].plot(t, gt_mono, alpha=0.5, label="Ground Truth", color="gray", linewidth=0.5)
    axes[idx].plot(t, est_mono[:n], alpha=0.5, label="Demucs", color=STEMS_COLOR[stem], linewidth=0.5)
    axes[idx].set_ylabel("Amplitude")
    axes[idx].set_title(f"{STEMS_RU[stem]}: GT vs Predicted")
    axes[idx].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 4. Анализ показателей (5 треков)

In [ ]:
results_dir = Path("evaluation_results")
df = pd.read_csv(results_dir / "metrics.csv")
print(df.head())
print(f"\nTotal entries: {len(df)}")

In [ ]:
# Итоговая статистика
summary = df.groupby("stem").agg({
    "SDR": ["mean", "std", "min", "max"],
    "SIR": ["mean", "std", "min", "max"],
    "SAR": ["mean", "std", "min", "max"],
})
print(summary)

In [ ]:
# Боксплот: SDR по стемам
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(["SDR", "SIR", "SAR"]):
    data = [df[df["stem"] == s][metric].values for s in STEMS]
    bp = axes[i].boxplot(data, labels=[STEMS_RU[s] for s in STEMS], patch_artist=True)
    for patch, s in zip(bp["boxes"], STEMS):
        patch.set_facecolor(STEMS_COLOR[s])
        patch.set_alpha(0.7)
    axes[i].set_ylabel(f"{metric} (dB)")
    axes[i].set_title(f"{metric} Distribution")
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Demucs htdemucs — Quality Metrics on MUSDB18-7 (5 tracks)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Столбчатая диаграмма: среднее значение SDR на стеме с погрешностями
means = df.groupby("stem")["SDR"].mean()
stds = df.groupby("stem")["SDR"].std()

fig, ax = plt.subplots(figsize=(8, 5))
names = [STEMS_RU[s] for s in STEMS]
colors = [STEMS_COLOR[s] for s in STEMS]

bars = ax.bar(names, means.values, yerr=stds.values, capsize=5, color=colors, alpha=0.7, edgecolor="white")
ax.axhline(y=7.0, color="red", linestyle="--", alpha=0.5, label="Expected ~7.0 dB")
ax.set_ylabel("SDR (dB)")
ax.set_title("Average SDR by Stem (mean ± std)")
ax.legend()
ax.grid(True, alpha=0.3)

# Добавить подписи значений
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.2f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Карта интенсивности по треку
pivot = df.pivot_table(index="track", columns="stem", values="SDR")
pivot.index = [t[:30] for t in pivot.index]  # truncate long names

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=15)
ax.set_xticks(range(len(STEMS)))
ax.set_xticklabels([STEMS_RU[s] for s in STEMS])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_title("SDR (dB) per Track and Stem")

# Добавить аннотации значений
for i in range(len(pivot.index)):
    for j in range(len(STEMS)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.1f}", ha="center", va="center",
               color="black" if 3 < val < 12 else "white", fontweight="bold")

plt.colorbar(im, ax=ax, label="SDR (dB)")
plt.tight_layout()
plt.show()

## 5. Выводы

| Metric | Vocals | Drums | Bass | Other | Overall |
|--------|--------|-------|------|-------|---------|
| **SDR** | 9.27 | 5.73 | 9.96 | 3.86 | 7.21 |
| **SIR** | 15.25 | 8.76 | 16.46 | 5.98 | 11.61 |
| **SAR** | 10.23 | 8.41 | 11.11 | 4.78 | 8.63 |

**Основные наблюдения:**
- **Разделение баса** самое сильное (SDR = 9.96 дБ), вероятно потому, что басовые частоты хорошо отделяются в спектральной области
- **Вокал** тоже показывает хорошие результаты (SDR = 9.27 дБ) благодаря гибридной трансформерной архитектуре
- **Барабаны** ают переменные результаты (SDR = 5.73 дБ в среднем) — треки с редкими ударными партиями разделяются хорошо, треки со сложными ритмами — труднее
- **Остальное**  самый слабый стем (SDR = 3.86 дБ), что ожидаемо — этот стем содержит смесь всех оставшихся инструментов
- Общий SDR 7.21 дБ — приемлемый результат для инференса одной моделью на CPU



**Примечания:**
- MUSDB18-7 содержит короткие отрывки по 6.8 секунд, которые могут не отражать поведение на полных треках
- Время обработки: ~6.5 с на 6.8-секундный трек на CPU (близко к реальному времени)
- С GPU (CUDA) скорость была бы в ~10–20 раз выше